In [1]:
import os
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import re
import openpyxl
import numpy as np

In [ ]:
folder=r'Path'
OFolder=r'Output Folder Path'

In [ ]:
file_list=[]
for (root, dirs, file) in os.walk(folder):
    for f in file:
        if ('.xlsm') in f:
                    file_list.append(f)
file_list

In [4]:
file_link=[]

for i in range(len(file_list)):
     for r,d,f in os.walk(folder):
          for files in f:
               if files == file_list[i]:
                    file_link.append(os.path.join(r,files))

In [5]:
len(file_link)

71

In [ ]:
file_link[1]

In [ ]:
cols=["Filename","part Type",'Product ID','*\xa0<Name>', 'Working Column','Supplier Classification Reference', 'Manufacturer', 'ASG Vendor Name','<Parent ID>', '<Object Type Name>','Attribute','value']
df_s = pd.DataFrame(columns=cols)
df_s


In [ ]:
repl='''folder\\'''
IDS=["Filename","part Type",'Product ID','*\xa0<Name>', 'Working Column','Supplier Classification Reference', 'Manufacturer', 'ASG Vendor Name','<Parent ID>', '<Object Type Name>']

In [ ]:
#trial to delete




for i in range(len(file_link)):
    workbook=openpyxl.load_workbook(file_link[i])
    sheetname=workbook.sheetnames
    print(sheetname)
    for s in sheetname:
        if (("STEP" not in s) and ("Cover" not in s)):
            print(s)
            df=pd.read_excel(file_link[i],sheet_name=s, skiprows=9)
            df["Filename"]=file_link[i]
            df=df[['Filename','<Parent ID>']].drop_duplicates()
            df_s=pd.concat([df_s,df])


In [ ]:
df_s

In [22]:
repl='''C:\\Users\\Vikram.Vadhirajan\\OneDrive - Trico\\Documents - Product Support India Romania\\Catalog\\Score card\\NAPA\\Mandatory\\Input\\'''
IDS=["part Type",'Product ID','*\xa0<Name>', 'Working Column', 'Manufacturer', '<Parent ID>', '<Object Type Name>']

In [ ]:
filelink=r'C:\Users\vikram.vadhirajan\OneDrive - Trico\FBG\99_Downloads\Excel--2026-01-13--05-40-2026-01-13_05.47.50.xlsm'    
workbook=openpyxl.load_workbook(filelink)
sheetname=workbook.sheetnames
print(sheetname)
for s in sheetname:
    if (("STEP" not in s) and ("Cover" not in s)):
        print(s)
        df=pd.read_excel(filelink,sheet_name=s, skiprows=9)
        # df["Filename"]=filelink.replace(repl,"")
        df["part Type"]=df['<Parent ID>'].mode()[0].split('''(''')[0]
        df=pd.melt(df,id_vars=IDS,var_name='Attribute').reset_index(drop=True)
        df_s=pd.concat([df_s,df])

In [ ]:
df_s

In [ ]:
df_parttype=df_s[["part Type", '*\xa0<Name>','Attribute', 'value']].drop_duplicates()
df_parttype=df_parttype[df_parttype['Attribute']=="Brand"]
df_parttype

In [ ]:
for i in range(1,len(file_link)):
    workbook=openpyxl.load_workbook(file_link[i])
    sheetname=workbook.sheetnames
    print(sheetname)
    for s in sheetname:
        if (("STEP" not in s) and ("Cover" not in s)):
            print(s)
            df=pd.read_excel(file_link[i],sheet_name=s, skiprows=9)
            df["Filename"]=file_link[i].replace(repl,"")
            df["part Type"]=df["part Type"]=df['<Parent ID>'].mode()[0].split('''(''')[0]
            df=pd.melt(df,id_vars=IDS).reset_index(drop=True)
            df_s=pd.concat([df_s,df])

In [ ]:
df_s.shape

In [ ]:
df_s

In [12]:
df_s=df_s[df_s['*\xa0<Name>']!="SAMPLE123"]

In [13]:
df_s['Brand']=df_s['Filename']
repllist={'Autolite - Smart Sheet 11.1.24.xlsm': "Autolite",'NAPA BPI Smartsheet Export 2024-10-22.xlsm': "BPI",'NAPA TRICO Smartsheet Export 2024-10-22.xlsm': "TRICO"}

df_s['Brand'] = df_s['Brand'].replace(repllist, regex=True)

In [42]:
df_s['Key']=df_s.Brand+df_s['part Type']+df_s['variable']

In [43]:
df_s=df_s.reset_index(drop=True)

In [44]:
df_untouched=df_s

In [45]:
df_untouched.shape

(744461, 14)

In [46]:
df_s=df_untouched[df_untouched['Key']!="BPIBrake HosePosition ☰      "]


In [47]:
df_s=df_s[~df_s.Key.str.contains(".1")]

In [ ]:
df_s

In [49]:
df_s=df_s.reset_index(drop=True)

In [50]:
for i in range (len(df_s)):
    s=df_s['Key'][i]
    clean = re.sub(r'[^a-zA-Z0-9]', '', s)
    df_s.loc[i,'Key']=clean

In [51]:
df_s['Filename'][1]

'Autolite - Smart Sheet 11.1.24.xlsm'

In [ ]:
df_s.head()

In [53]:
df_s.shape

(708044, 14)

In [ ]:
df_attribute=pd.read_excel(r"path/filename.xlsx",skiprows=2)

In [55]:
for i in range (len(df_attribute)):
    s=df_attribute['Key'][i]
    clean = re.sub(r'[^a-zA-Z0-9]', '', s)
    df_attribute.loc[i,'Key']=clean

In [ ]:
df_attribute['Key'].value_counts()

In [ ]:
df_final=pd.merge(df_s,df_attribute[['Key','Required']], on='Key', how='left')
df_final.head()

In [58]:
df_final.shape

(708044, 15)

In [ ]:
df_final['Key'].value_counts()

In [ ]:
df_final.columns

In [61]:
df_List=df_final[[ 'part Type', 'Product ID', '*\xa0<Name>',  '<Parent ID>', 'variable', 'value', 'Brand','Required']]

In [62]:
df_List=df_List.rename(columns={'part Type': "Part Type", '*\xa0<Name>': "SKU",  'variable': "Attribute", 'value':"Value"})

In [ ]:
df_List[['Brand','Part Type', 'SKU','Product ID',  '<Parent ID>',  'Required','Attribute', 'Value']]

In [ ]:
# for i in range(len(df_List)):
#     s=df_List["Values"][i]
#     if s is np.nan:
#         df_List.loc[i,'Values']=0
#     else:
#         df_List.loc[i,'Values']=1

In [318]:
#df_List=df_List.dropna()

In [ ]:
#df_List=df_List[df_List["Required"]=="Mandatory"]

In [64]:
df_List.shape

(708044, 8)

In [65]:
df_List=df_List[['Brand','Part Type', 'SKU','Product ID',  '<Parent ID>',  'Required','Attribute', 'Value']]

In [66]:
with pd.ExcelWriter(OFolder+'\\'+'NAPA_Mandatory_Baseline_updated.xlsx') as writer:  # doctest: +SKIP
    df_List.to_excel(writer,index=False, sheet_name='Raw')